# Modelo YOLO unificado para objetos urbanos

Esta célula prepara o ambiente, monta o Google Drive, define a configuração global e obtém a chave da API da Ultralytics com entrada oculta como último recurso. Os diretórios públicos em /content são vínculos para /content/local_work, mantendo downloads, conversões, deduplicação e treinamento no armazenamento local.

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "ultralytics", "nest-asyncio", "onnx", "onnxslim"],
    check=True,
)

import asyncio
import getpass
import hashlib
import os
import shutil
from pathlib import Path

import nest_asyncio
import yaml
from google.colab import drive, userdata
from ultralytics import YOLO
from ultralytics.data.converter import convert_ndjson_to_yolo
from ultralytics.utils.checks import check_file

nest_asyncio.apply()

try:
    drive.mount("/content/drive")
except Exception:
    drive._mount("/content/drive")

DATASET_1_URI = "ul://baguette/datasets/arquitetura"
DATASET_2_URI = "ul://baguette/datasets/asfalto"
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/LIA/estacionamento"
SAMPLE_FRACTION = 1.0
MODEL_BASE = "yolo26n.pt"
EPOCHS = 20
BATCH_SIZE = 16
IMGSZ = 640
ONNX_DYNAMIC = True

OUTPUT_DIR = Path(DRIVE_OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

api_key = os.environ.get("ULTRALYTICS_API_KEY", "").strip()
if not api_key:
    try:
        api_key = (userdata.get("ULTRALYTICS_API_KEY") or "").strip()
    except Exception:
        api_key = ""
if not api_key:
    api_key = getpass.getpass("ULTRALYTICS_API_KEY: ").strip()
if not api_key:
    raise RuntimeError("ULTRALYTICS_API_KEY não pode estar vazia.")
os.environ["ULTRALYTICS_API_KEY"] = api_key

LOCAL_WORK = Path("/content/local_work")
LOCAL_NDJSON = LOCAL_WORK / "ndjson"
LOCAL_UNIFIED = LOCAL_WORK / "unified_dataset"
PUBLIC_NDJSON = Path("/content/ndjson")
UNIFIED_ROOT = Path("/content/unified_dataset")
LOCAL_WORK.mkdir(parents=True, exist_ok=True)

def bind_local_directory(public_path, local_path):
    local_path.mkdir(parents=True, exist_ok=True)
    if public_path.is_symlink():
        if public_path.resolve() != local_path.resolve():
            public_path.unlink()
            public_path.symlink_to(local_path, target_is_directory=True)
    elif public_path.exists():
        if public_path.resolve() != local_path.resolve():
            for source_item in public_path.iterdir():
                destination_item = local_path / source_item.name
                if not destination_item.exists():
                    shutil.move(str(source_item), str(destination_item))
            shutil.rmtree(public_path)
            public_path.symlink_to(local_path, target_is_directory=True)
    else:
        public_path.symlink_to(local_path, target_is_directory=True)

bind_local_directory(PUBLIC_NDJSON, LOCAL_NDJSON)
bind_local_directory(UNIFIED_ROOT, LOCAL_UNIFIED)

DRIVE_ARCHIVE = OUTPUT_DIR / "dataset_unified.zip"
DRIVE_ONNX = OUTPUT_DIR / "model_best.onnx"
DRIVE_BEST = OUTPUT_DIR / "best.pt"
target_artifacts = (DRIVE_ARCHIVE, DRIVE_ONNX, DRIVE_BEST)

FORCE_REBUILD = False
if any(path.exists() for path in target_artifacts):
    FORCE_REBUILD = input("Artifacts found in Google Drive. Overwrite and re-process everything? (y/N): ").strip().lower() == "y"

UNIFIED_YAML = UNIFIED_ROOT / "data.yaml"
dataset_cache_ready = (
    UNIFIED_YAML.is_file()
    and (UNIFIED_ROOT / "images" / "train").is_dir()
    and (UNIFIED_ROOT / "images" / "val").is_dir()
)
REBUILD_DATASET = FORCE_REBUILD or not dataset_cache_ready
print(f"Reconstrução do dataset: {REBUILD_DATASET}")

## Funções de leitura, conversão e deduplicação

As funções desta célula interpretam data.yaml em seus formatos usuais, encontram imagens e rótulos, calculam SHA-256 e convertem caixas ou polígonos em caixas YOLO. IDs de classe são remapeados e todas as coordenadas finais são limitadas ao intervalo fechado de 0 a 1.

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

def load_dataset_yaml(data_yaml_path):
    data_yaml_path = Path(data_yaml_path)
    with data_yaml_path.open("r", encoding="utf-8") as stream:
        dataset_data = yaml.safe_load(stream) or {}
    raw_names = dataset_data.get("names", {})
    if isinstance(raw_names, list):
        raw_names = {index: name for index, name in enumerate(raw_names)}
    if not isinstance(raw_names, dict) or not raw_names:
        raise ValueError(f"Classes ausentes ou inválidas em {data_yaml_path}")
    names = {int(class_id): str(name).strip().lower() for class_id, name in raw_names.items()}
    return dataset_data, names

def resolve_split_sources(dataset_data, data_yaml_path, split_name):
    split_definition = dataset_data.get(split_name)
    if split_definition is None:
        raise ValueError(f"Split {split_name} ausente em {data_yaml_path}")
    declared_root = dataset_data.get("path")
    if declared_root:
        dataset_root = Path(declared_root)
        if not dataset_root.is_absolute():
            dataset_root = Path(data_yaml_path).parent / dataset_root
    else:
        dataset_root = Path(data_yaml_path).parent
    definitions = split_definition if isinstance(split_definition, list) else [split_definition]
    sources = []
    for definition in definitions:
        source = Path(definition)
        if not source.is_absolute():
            source = dataset_root / source
        sources.append(source.resolve())
    return sources

def enumerate_images(split_source):
    split_source = Path(split_source)
    if split_source.is_dir():
        return sorted(
            path for path in split_source.rglob("*")
            if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
        )
    if split_source.is_file() and split_source.suffix.lower() == ".txt":
        images = []
        with split_source.open("r", encoding="utf-8") as stream:
            for raw_path in stream:
                image_path = Path(raw_path.strip())
                if not image_path.is_absolute():
                    image_path = split_source.parent / image_path
                if image_path.is_file() and image_path.suffix.lower() in IMAGE_EXTENSIONS:
                    images.append(image_path.resolve())
        return sorted(images)
    return []

def corresponding_label_path(image_path):
    image_path = Path(image_path)
    parts = list(image_path.parts)
    indexes = [index for index, part in enumerate(parts) if part == "images"]
    if indexes:
        image_index = indexes[-1]
        return Path(*parts[:image_index], "labels", *parts[image_index + 1:]).with_suffix(".txt")
    return image_path.parent.parent / "labels" / f"{image_path.stem}.txt"

def compute_sha256(file_path):
    digest = hashlib.sha256()
    with Path(file_path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def clamp_unit(value):
    return max(0.0, min(1.0, float(value)))

def convert_label_file(label_path, class_mapping):
    label_path = Path(label_path)
    if not label_path.is_file():
        return ""
    converted_lines = []
    with label_path.open("r", encoding="utf-8") as stream:
        for raw_line in stream:
            tokens = raw_line.strip().split()
            if len(tokens) < 5:
                continue
            try:
                source_class = int(float(tokens[0]))
                coordinates = [float(token) for token in tokens[1:]]
            except ValueError:
                continue
            if source_class not in class_mapping:
                continue
            remapped_class = class_mapping[source_class]
            if len(tokens) == 5:
                x_center, y_center, width, height = [clamp_unit(value) for value in coordinates]
            else:
                if len(coordinates) % 2 != 0:
                    coordinates = coordinates[:-1]
                if len(coordinates) < 6:
                    continue
                x_values = coordinates[0::2]
                y_values = coordinates[1::2]
                x_min, x_max = min(x_values), max(x_values)
                y_min, y_max = min(y_values), max(y_values)
                x_center = clamp_unit((x_min + x_max) / 2.0)
                y_center = clamp_unit((y_min + y_max) / 2.0)
                width = clamp_unit(x_max - x_min)
                height = clamp_unit(y_max - y_min)
            converted_lines.append(
                f"{remapped_class} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}\n"
            )
    return "".join(converted_lines)

## Download, conversão e consolidação global

Quando não existe cache local válido, os dois NDJSON são baixados e convertidos. As classes normalizadas recebem IDs globais sequenciais. Cada imagem ganha o prefixo ds1_ ou ds2_. Duplicatas no mesmo split têm anotações anexadas ao rótulo canônico; hashes presentes nos dois splits são eliminados para evitar vazamento entre treino e validação.

In [ ]:
if REBUILD_DATASET:
    if LOCAL_NDJSON.exists():
        shutil.rmtree(LOCAL_NDJSON)
    LOCAL_NDJSON.mkdir(parents=True, exist_ok=True)

    conversion_dir_1 = LOCAL_WORK / "converted_dataset_1"
    conversion_dir_2 = LOCAL_WORK / "converted_dataset_2"
    for conversion_dir in (conversion_dir_1, conversion_dir_2):
        if conversion_dir.exists():
            shutil.rmtree(conversion_dir)

    ndjson_path_1 = Path(check_file(DATASET_1_URI, download_dir="/content/ndjson"))
    ndjson_path_2 = Path(check_file(DATASET_2_URI, download_dir="/content/ndjson"))
    data_yaml_1 = Path(
        asyncio.run(
            convert_ndjson_to_yolo(
                ndjson_path_1,
                output_path=conversion_dir_1,
                fraction=SAMPLE_FRACTION,
            )
        )
    )
    data_yaml_2 = Path(
        asyncio.run(
            convert_ndjson_to_yolo(
                ndjson_path_2,
                output_path=conversion_dir_2,
                fraction=SAMPLE_FRACTION,
            )
        )
    )
    if not data_yaml_1.is_file() or not data_yaml_2.is_file():
        raise FileNotFoundError("A conversão não retornou dois arquivos data.yaml válidos.")

    dataset_data_1, names_1 = load_dataset_yaml(data_yaml_1)
    dataset_data_2, names_2 = load_dataset_yaml(data_yaml_2)
    unified_name_to_id = {}
    unified_names = {}
    source_mapping_1 = {}
    source_mapping_2 = {}

    for source_id, normalized_name in sorted(names_1.items()):
        if normalized_name not in unified_name_to_id:
            unified_id = len(unified_name_to_id)
            unified_name_to_id[normalized_name] = unified_id
            unified_names[unified_id] = normalized_name
        source_mapping_1[source_id] = unified_name_to_id[normalized_name]

    for source_id, normalized_name in sorted(names_2.items()):
        if normalized_name not in unified_name_to_id:
            unified_id = len(unified_name_to_id)
            unified_name_to_id[normalized_name] = unified_id
            unified_names[unified_id] = normalized_name
        source_mapping_2[source_id] = unified_name_to_id[normalized_name]

    if LOCAL_UNIFIED.exists():
        shutil.rmtree(LOCAL_UNIFIED)
    LOCAL_UNIFIED.mkdir(parents=True, exist_ok=True)
    for split_name in ("train", "val"):
        (UNIFIED_ROOT / "images" / split_name).mkdir(parents=True, exist_ok=True)
        (UNIFIED_ROOT / "labels" / split_name).mkdir(parents=True, exist_ok=True)

    image_hashes = {}
    leakage_hashes = set()
    used_names = {"train": set(), "val": set()}
    copied_images = 0
    merged_duplicates = 0
    leakage_removals = 0
    dataset_specs = (
        ("ds1", dataset_data_1, data_yaml_1, source_mapping_1),
        ("ds2", dataset_data_2, data_yaml_2, source_mapping_2),
    )

    for prefix, dataset_data, source_yaml, class_mapping in dataset_specs:
        for split_name in ("train", "val"):
            split_sources = resolve_split_sources(dataset_data, source_yaml, split_name)
            for source_index, split_source in enumerate(split_sources, start=1):
                for source_image in enumerate_images(split_source):
                    relative_name = source_image.name
                    if split_source.is_dir():
                        relative_name = "__".join(source_image.relative_to(split_source).parts)
                    destination_name = f"{prefix}_{relative_name}"
                    if destination_name in used_names[split_name]:
                        destination_name = f"{prefix}_root{source_index}_{relative_name}"
                    collision_number = 2
                    collision_base = destination_name
                    while destination_name in used_names[split_name]:
                        destination_name = (
                            f"{Path(collision_base).stem}_{collision_number}{Path(collision_base).suffix}"
                        )
                        collision_number += 1
                    used_names[split_name].add(destination_name)

                    destination_image = UNIFIED_ROOT / "images" / split_name / destination_name
                    destination_stem = destination_image.stem
                    destination_label = UNIFIED_ROOT / "labels" / split_name / f"{destination_stem}.txt"
                    annotation_text = convert_label_file(
                        corresponding_label_path(source_image), class_mapping
                    )
                    shutil.copy2(source_image, destination_image)
                    image_hash = compute_sha256(destination_image)

                    if image_hash in leakage_hashes:
                        destination_image.unlink()
                        continue

                    if image_hash in image_hashes:
                        canonical_split, canonical_stem = image_hashes[image_hash]
                        destination_image.unlink()
                        if canonical_split == split_name:
                            canonical_label = (
                                UNIFIED_ROOT / "labels" / canonical_split / f"{canonical_stem}.txt"
                            )
                            if annotation_text:
                                with canonical_label.open("a", encoding="utf-8") as stream:
                                    stream.write(annotation_text)
                            merged_duplicates += 1
                        else:
                            canonical_images = list(
                                (UNIFIED_ROOT / "images" / canonical_split).glob(f"{canonical_stem}.*")
                            )
                            for canonical_image in canonical_images:
                                if canonical_image.is_file():
                                    canonical_image.unlink()
                            canonical_label = (
                                UNIFIED_ROOT / "labels" / canonical_split / f"{canonical_stem}.txt"
                            )
                            if canonical_label.exists():
                                canonical_label.unlink()
                            del image_hashes[image_hash]
                            leakage_hashes.add(image_hash)
                            leakage_removals += 1
                        continue

                    image_hashes[image_hash] = (split_name, destination_stem)
                    with destination_label.open("w", encoding="utf-8") as stream:
                        stream.write(annotation_text)
                    copied_images += 1

    unified_yaml_data = {
        "path": str(UNIFIED_ROOT),
        "train": "images/train",
        "val": "images/val",
        "names": unified_names,
    }
    with UNIFIED_YAML.open("w", encoding="utf-8") as stream:
        yaml.safe_dump(unified_yaml_data, stream, sort_keys=False, allow_unicode=True)

    remaining_images = sum(
        1
        for split_name in ("train", "val")
        for image_path in (UNIFIED_ROOT / "images" / split_name).iterdir()
        if image_path.is_file()
    )
    print(f"Classes unificadas: {unified_names}")
    print(f"Imagens copiadas antes de remoções cruzadas: {copied_images}")
    print(f"Duplicatas mescladas no mesmo split: {merged_duplicates}")
    print(f"Hashes eliminados por vazamento: {leakage_removals}")
    print(f"Imagens finais: {remaining_images}")
else:
    print(f"Dataset local reutilizado: {UNIFIED_YAML}")

## Backup do dataset e treinamento

O dataset é compactado localmente em /content/dataset_unified.zip e apenas os artefatos finais são copiados para o Drive. Caso best.pt já exista e a reconstrução não tenha sido forçada, o usuário decide se deseja treinar novamente. O modelo yolo11n.pt é usado se o modelo-base solicitado não puder ser resolvido.

In [ ]:
if FORCE_REBUILD or not DRIVE_ARCHIVE.exists():
    local_archive = Path(
        shutil.make_archive(
            "/content/dataset_unified",
            "zip",
            root_dir=str(LOCAL_WORK),
            base_dir="unified_dataset",
        )
    )
    shutil.copy2(local_archive, DRIVE_ARCHIVE)
    shutil.copy2(UNIFIED_YAML, OUTPUT_DIR / "data.yaml")
    print(f"Dataset arquivado em: {DRIVE_ARCHIVE}")
else:
    print(f"Backup existente reutilizado: {DRIVE_ARCHIVE}")

RETRAIN_MODEL = True
if DRIVE_BEST.exists() and not FORCE_REBUILD:
    RETRAIN_MODEL = input("best.pt found in Google Drive. Retrain the model? (y/N): ").strip().lower() == "y"

LOCAL_BEST = LOCAL_WORK / "best.pt"
if RETRAIN_MODEL:
    try:
        model = YOLO(MODEL_BASE)
    except Exception as model_error:
        print(f"Falha ao carregar {MODEL_BASE}: {model_error}")
        print("Usando o fallback yolo11n.pt.")
        model = YOLO("yolo11n.pt")

    model.train(
        data=str(UNIFIED_YAML),
        epochs=EPOCHS,
        batch=BATCH_SIZE,
        imgsz=IMGSZ,
        project=str(LOCAL_WORK / "runs"),
        name="urban_detection",
        exist_ok=True,
    )
    trained_best = Path(str(model.trainer.best))
    if not trained_best.is_file():
        raise FileNotFoundError(f"Melhor checkpoint não encontrado: {trained_best}")
    shutil.copy2(trained_best, LOCAL_BEST)
    best_pt = LOCAL_BEST
    shutil.copy2(best_pt, DRIVE_BEST)

    training_output_dir = Path(model.trainer.save_dir)
    for artifact_name in ("confusion_matrix.png", "results.png"):
        source_artifact = training_output_dir / artifact_name
        if not source_artifact.is_file():
            raise FileNotFoundError(f"Artefato de treinamento não encontrado: {source_artifact}")
        shutil.copy2(source_artifact, OUTPUT_DIR / artifact_name)
else:
    shutil.copy2(DRIVE_BEST, LOCAL_BEST)
    best_pt = LOCAL_BEST
    print(f"Checkpoint reutilizado localmente: {best_pt}")

## Validação, inferência e exportação ONNX

O melhor checkpoint é validado no split val. Uma imagem é usada para imprimir caixas, confianças e nomes de classe. Por fim, o modelo é simplificado e exportado para ONNX, copiado como model_best.onnx e acompanhado por um resumo dos arquivos preservados no Drive.

In [ ]:
eval_model = YOLO(best_pt)
validation_metrics = eval_model.val(
    data=str(UNIFIED_YAML),
    split="val",
    imgsz=IMGSZ,
    batch=BATCH_SIZE,
)
print(f"mAP50: {validation_metrics.box.map50:.6f}")
print(f"mAP50-95: {validation_metrics.box.map:.6f}")

validation_images = sorted(
    path
    for path in (UNIFIED_ROOT / "images" / "val").iterdir()
    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
)
if not validation_images:
    raise RuntimeError("Não há imagens de validação para o teste de inferência.")

sample_image = validation_images[0]
inference_result = eval_model.predict(source=str(sample_image), imgsz=IMGSZ, verbose=False)[0]
print(f"Imagem de teste: {sample_image}")
if inference_result.boxes is None or len(inference_result.boxes) == 0:
    print("Nenhuma detecção encontrada.")
else:
    for detected_box in inference_result.boxes:
        class_id = int(detected_box.cls.item())
        confidence = float(detected_box.conf.item())
        class_name = inference_result.names.get(class_id, str(class_id))
        xyxy = [round(float(value), 2) for value in detected_box.xyxy[0].tolist()]
        print(f"classe={class_name}, confiança={confidence:.4f}, xyxy={xyxy}")

onnx_path = eval_model.export(
    format="onnx",
    dynamic=ONNX_DYNAMIC,
    simplify=True,
    imgsz=IMGSZ
)
onnx_path = Path(onnx_path)
if not onnx_path.is_file():
    raise FileNotFoundError(f"Arquivo ONNX não encontrado: {onnx_path}")
shutil.copy2(onnx_path, DRIVE_ONNX)

print(f"Artefatos salvos em {OUTPUT_DIR}:")
for artifact_path in sorted(OUTPUT_DIR.iterdir()):
    if artifact_path.is_file():
        print(f"- {artifact_path.name} ({artifact_path.stat().st_size:,} bytes)")